In [25]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.model_selection import cross_val_score

import pandas as pd

from pm import config, data, split, models


df = data.load_processed()

train, _test = split.split_train_test(df)

feature_columns = config.BASIC_NUMERICAL_COLUMNS + config.PHYSICAL_NUMERICAL_COLUMNS + config.CATEGORICAL_COLUMNS

X = train[feature_columns]

y = train[config.LABEL]

groups = train[config.GROUPS]


In [28]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=config.SEED)

column_sets = {
    "basic": config.BASIC_NUMERICAL_COLUMNS,
    "basic + physical": config.BASIC_NUMERICAL_COLUMNS + config.PHYSICAL_NUMERICAL_COLUMNS,
}

results = []

for set_name, numeric_columns in column_sets.items():
    for class_weight in [None, "balanced"]:
        pipeline = models.build_logistic_pipeline(numeric_columns, class_weight=class_weight)

        scores = cross_val_score(pipeline, X, y, cv=sgkf, groups=groups,
                                 scoring="average_precision")

        results.append({
            "columns": set_name,
            "class_weight": class_weight,
            "pr_auc_mean": scores.mean(),
            "pr_auc_std": scores.std(),
            "scores": scores
        })

results = pd.DataFrame(results)
print(f"Random baseline (failure rate): {y.mean():.3f}")
print(results.round(3))

Random baseline (failure rate): 0.034
            columns class_weight  pr_auc_mean  pr_auc_std  \
0             basic          NaN        0.464       0.052   
1             basic     balanced        0.427       0.055   
2  basic + physical          NaN        0.494       0.045   
3  basic + physical     balanced        0.445       0.041   

                                              scores  
0  [0.395645, 0.515101, 0.523787, 0.471417, 0.412...  
1   [0.3424, 0.478033, 0.491903, 0.428054, 0.392747]  
2  [0.47649, 0.523717, 0.535694, 0.521068, 0.412857]  
3  [0.406883, 0.430305, 0.481073, 0.505719, 0.402...  
